# Documentation Q&A over Wikibooks — Walkthrough

A documentation chatbot built the way a company-docs assistant is built: **ingest** manuals,
**index** them two ways, **retrieve** with six methods, **answer with citations** (or say
*"Not in docs"*), expose it all as an **MCP server**, and **measure** every step.

The "documentation" is three Wikibooks manuals: *Python Programming*, *Structured Query Language*
and *Git* (CC BY-SA 4.0). Every output below is real: produced by running this notebook against the
built index and the saved evaluation results.

**Sections:** 1. Data and ingestion · 2. Indexing · 3. Retrieval, six methods · 4. Answering with
citations · 5. The MCP server · 6. Evaluation · 7. Findings

Prerequisites: `python build_index.py` (downloads + indexes, ~20 min the first time) and
`python eval/run_eval.py` (writes `eval/results/`). Live Claude calls in this notebook: about 6.

In [1]:
import json, os, subprocess, sys, warnings
from pathlib import Path
from statistics import median

import logging
warnings.filterwarnings("ignore")
os.environ["TQDM_DISABLE"] = "1"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from docsrag.config import BOOKS, USAGE
from docsrag.ingest import load_raw, split_sections, chunk_section

def table(rows, headers):
    """Render a list of rows as a Markdown table."""
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))

## 1. The data and ingestion

`build_index.py` downloaded every page of the three books through the Wikibooks API as **plain
text with `== headings ==`**, one page per request, waiting whenever the API rate-limited it. Pages
under 200 characters (navigation stubs) and "Print version" pages (which duplicate whole books) were dropped.

In [2]:
rows = []
for key in BOOKS:
    pages = load_raw(key)
    rows.append([BOOKS[key].replace("_", " "), len(pages), f"{sum(len(p['text']) for p in pages):,}"])
table(rows, ["Book", "Pages kept", "Characters"])

| Book | Pages kept | Characters |
|---|---|---|
| Python Programming | 90 | 385,688 |
| Structured Query Language | 59 | 266,995 |
| Git | 22 | 81,279 |

The text is small (well under 1 MB). One real page, *Python Programming / Lists*:

In [3]:
page = next(p for p in load_raw("python") if p["title"] == "Python Programming/Lists")
print(f"{page['title']}  |  {len(page['text']):,} characters  |  revision {page['revid']}\n")
print(page["text"][:500])

Python Programming/Lists  |  19,858 characters  |  revision 4636167

A list in Python is an ordered group of items (or elements). It is a very general structure, and list elements don't have to be of the same type: you can put numbers, letters, strings and nested lists all on the same list.


== Overview ==
Lists in Python at a glance:

list1 = []                      # A new empty list
list2 = [1, 2, 3, "cat"]        # A new non-empty list with mixed item types
list1.append("cat")             # Add a single member, at the end of the list
list1.extend(["dog", "mo


### Split at headings → sections

Each `== Heading ==` starts a **section**; sub-headings keep their parent in the path. A section is
the unit the chatbot **cites** and the evaluation **scores**, so it gets a stable id
`book/page#heading-path`. *See also / References / External links* and sections under 80 characters
are dropped.

Why headings and not fixed-size pieces? The authors already split each topic into meaningful
parts; cutting every N characters would cut examples in half.

In [4]:
sections = split_sections("python", page)
print(f"{len(sections)} sections\n")
table([[len(s["text"]), s["heading"], f"`{s['id']}`"] for s in sections[:10]],
      ["Chars", "Heading path", "Section id"])

26 sections



| Chars | Heading path | Section id |
|---|---|---|
| 222 | Introduction | `python/Lists#introduction` |
| 1922 | Overview | `python/Lists#overview` |
| 156 | List creation | `python/Lists#list-creation` |
| 464 | List creation > Plain creation | `python/Lists#list-creation-plain-creation` |
| 2081 | List creation > List comprehensions | `python/Lists#list-creation-list-comprehensions` |
| 1667 | List creation > List creation shortcuts | `python/Lists#list-creation-list-creation-shortcuts` |
| 112 | List size | `python/Lists#list-size` |
| 682 | Combining lists | `python/Lists#combining-lists` |
| 1718 | Getting pieces of lists (slices) > Continuous slices | `python/Lists#getting-pieces-of-lists-slices-continuous-slices` |
| 367 | Getting pieces of lists (slices) > Non-Continuous slices | `python/Lists#getting-pieces-of-lists-slices-non-continuous-slices` |

### Cut long sections → chunks

Sections up to ~1,170 characters stay **one chunk**. Longer ones are cut at blank lines into
~900-character chunks; each new chunk repeats the last ~150 characters of the previous one
(**overlap**, starting at a sentence boundary) so a sentence on a cut is still findable. Every chunk
starts with its location: a **contextual chunk header**, so search knows the chunk is about
*Python lists* even if its own text never says so.

In [5]:
sec = next(s for s in sections if s["heading"].endswith("List comprehensions"))
chunks = chunk_section(sec)
print(f"Section '{sec['heading']}': {len(sec['text'])} chars -> {len(chunks)} chunks\n")
for i, c in enumerate(chunks, 1):
    header, body = c.split("\n", 1)
    print(f"chunk {i} ({len(c)} chars)\n  header: {header}\n  starts: {body[:90]!r}\n")

Section 'List creation > List comprehensions': 2081 chars -> 3 chunks

chunk 1 (865 chars)
  header: Python Programming > Lists > List creation > List comprehensions
  starts: 'Using list comprehension, you describe the process using which the list should be created.'

chunk 2 (763 chars)
  header: Python Programming > Lists > List creation > List comprehensions
  starts: 'It will evaluate the items in all of the objects sequentially and will\nloop over the short'

chunk 3 (932 chars)
  header: Python Programming > Lists > List creation > List comprehensions
  starts: "['cp', 'ct', 'ap', 'at']\n>>> print([x+y for x in 'cat' for y in 'pot' if x != 't' or y != "



In [6]:
rows = []
for key in BOOKS:
    secs = [s for p in load_raw(key) for s in split_sections(key, p)]
    chs = [c for s in secs for c in chunk_section(s)]
    lens = [len(c) for c in chs]
    rows.append([key, len(secs), len(chs), min(lens), int(median(lens)), max(lens)])
table(rows, ["Book", "Sections", "Chunks", "Min chars", "Median chars", "Max chars"])

| Book | Sections | Chunks | Min chars | Median chars | Max chars |
|---|---|---|---|---|---|
| python | 475 | 669 | 70 | 609 | 2261 |
| sql | 332 | 441 | 143 | 666 | 1758 |
| git | 119 | 151 | 108 | 559 | 1959 |

## 2. Indexing: two ways to search

Every chunk is indexed **twice**:

- **Vector (meaning):** a local model (`all-MiniLM-L6-v2`; Claude has no embedding model) turns each
  chunk into **384 numbers**. Similar meaning → similar numbers, even with different words.
- **BM25 (keywords):** word counts per chunk, weighted by how rare each word is.

No vector database: 1,261 × 384 numbers fit in memory, and one matrix multiply compares a question
with every chunk in milliseconds.

In [7]:
from docsrag.index import Index, embed, tokenize
ix = Index.load()
print(f"{len(ix.sections)} sections, {len(ix.chunks)} chunks")
print(f"embeddings: {ix.emb.shape[0]} x {ix.emb.shape[1]} numbers  ({ix.emb.nbytes/1e6:.1f} MB)")
print("first 8 numbers of chunk 0:", ix.emb[0][:8].round(3))

926 sections, 1261 chunks
embeddings: 1261 x 384 numbers  (1.9 MB)
first 8 numbers of chunk 0: [-0.08  -0.006 -0.045  0.039 -0.035 -0.131 -0.018  0.079]


Meaning, not words: *undo a commit* is close to *revert the last change* (no shared words) and far from *sort a list*:

In [8]:
phrases = ["undo a commit", "revert the last change", "sort a list in Python", "bake sourdough bread"]
v = embed(phrases)
table([[p, f"{float(v[0] @ v[i]):.2f}"] for i, p in enumerate(phrases)], ["Phrase", "Similarity to 'undo a commit'"])

| Phrase | Similarity to 'undo a commit' |
|---|---|
| undo a commit | 1.00 |
| revert the last change | 0.53 |
| sort a list in Python | 0.02 |
| bake sourdough bread | 0.09 |

BM25 weights rare words higher. For one question, the rarity (`idf`) of each kept word:

In [9]:
q = "How do I undo my last commit but keep my changes?"
toks = tokenize(q)
print("tokens kept:", toks)
table([[t, f"{ix.bm25.idf.get(t, 0):.2f}"] for t in toks], ["Word", "Rarity (idf)"])

tokens kept: ['undo', 'my', 'last', 'commit', 'but', 'keep', 'my', 'changes']


| Word | Rarity (idf) |
|---|---|
| undo | 6.73 |
| my | 3.98 |
| last | 2.88 |
| commit | 2.70 |
| but | 1.48 |
| keep | 3.98 |
| my | 3.98 |
| changes | 2.96 |

## 3. Retrieval: six methods

All six return ranked **sections**, found through their chunks (search small, answer big):

| Method | How |
|---|---|
| `vector` | embed the question; cosine similarity with every chunk |
| `bm25` | keyword scoring |
| `hybrid` | vector + BM25 merged with **Reciprocal Rank Fusion** |
| `hybrid_rerank` | hybrid top 20 re-scored by a local **cross-encoder** reading question + chunk together |
| `multi_query` | Claude writes 3 rephrasings; hybrid for each; fused |
| `hyde` | Claude writes a hypothetical answer; search with it, fused with the question's results |

One real question, *"How do I undo my last commit but keep my changes?"*. The section that answers
it is **Git > Introduction > Removing files > Resetting a commit** (`git reset` "cancels the
commit(s) and leaves the file(s) uncommitted").

In [10]:
from docsrag.retrieve import search, METHODS
GOLD = "git/Introduction#removing-files-resetting-a-commit"

def show(query, method, k=5):
    ranked = search(query, method, k=10)
    rank = ranked.index(GOLD) + 1 if GOLD in ranked else "not in top 10"
    print(f"{method:14} gold rank: {rank}")
    for i, sid in enumerate(ranked[:k], 1):
        s = ix.sections[sid]
        print(f"   {i}. {'✓ ' if sid == GOLD else ''}{s['book_title']} > {s['page']} > {s['heading']}")

for m in ["vector", "bm25", "hybrid", "hybrid_rerank"]:
    show(q, m)

vector         gold rank: 2
   1. Git > Introduction > Removing files > Throwing away local, uncommitted changes
   2. ✓ Git > Introduction > Removing files > Resetting a commit
   3. Git > Introduction > Removing files > Reverting a commit
   4. Git > Introduction > Conclusion
   5. Git > Introduction > Adding and committing files > reset
bm25           gold rank: not in top 10
   1. Git > Introduction > Removing files
   2. Git > git-svn > Dealing with local changes
   3. Git > Introduction > Removing files > Throwing away local, uncommitted changes
   4. Structured Query Language > COMMIT and ROLLBACK > Introduction
   5. Git > Introduction > Adding and committing files
hybrid         gold rank: not in top 10
   1. Git > Introduction > Removing files > Throwing away local, uncommitted changes
   2. Git > Introduction > Removing files
   3. Git > Introduction > Removing files > Reverting a commit
   4. Structured Query Language > COMMIT and ROLLBACK > Introduction
   5. Git > Introdu

hybrid_rerank  gold rank: 7
   1. Git > Introduction > Removing files > Throwing away local, uncommitted changes
   2. Git > Gerrit Code Review > Use > Submitting changes for review > Re-submitting a changeset
   3. Git > git-svn > Dealing with local changes
   4. Structured Query Language > COMMIT and ROLLBACK > Introduction
   5. Git > Rebasing > Principle


**What happened:** vector search understands *undo* ≈ *cancel* and ranks the right section high.
BM25 misses it: the question's words aren't in the text. Fusion rewards sections that **both** lists
found, so the vector-only hit sinks; re-ranking lifts it but not into the top 5. One example proves
nothing, which is why section 6 measures 42 questions.

### Reciprocal Rank Fusion, and the book's bug

RRF scores each item `1 / (60 + rank)` in each list and adds them up. It must be keyed by the
**item**. The LangChain book's chapter 10 version keyed by *position*, so the same document found by
two lists was never merged:

In [11]:
from docsrag.retrieve import rrf
list_a, list_b = ["A", "B", "X"], ["C", "B", "Y"]   # B is 2nd in both lists

def rrf_by_position(*lists, k=60):                  # the book's version: key = (list, position)
    scores = {(i, r): 1/(k + r + 1) for i, l in enumerate(lists) for r, _ in enumerate(l)}
    order = sorted(scores, key=scores.get, reverse=True)
    return [lists[i][r] for i, r in order]

print("by position (bug):", rrf_by_position(list_a, list_b))
print("by item (fixed):  ", rrf(list_a, list_b))

by position (bug): ['A', 'C', 'B', 'B', 'X', 'Y']
by item (fixed):   ['B', 'A', 'C', 'X', 'Y']


With the bug, **B** appears twice and never beats the single hits; fixed, B is merged and ranks first.

### Multi-query and HyDE (live Claude calls)

In [12]:
from docsrag.retrieve import _rephrasings, _hypothetical_answer
print("Rephrasings:"); [print("  -", r) for r in _rephrasings(q)]
print("\nHypothetical answer (HyDE):\n ", _hypothetical_answer(q)[:400])
print()
for m in ["multi_query", "hyde"]:
    show(q, m)

Rephrasings:


  - How can I revert my most recent commit in Git while preserving the working directory changes?
  - What's the proper way to reset HEAD to the previous commit with a soft reset so my staged modifications remain intact?
  - How do I roll back a Git commit without discarding the uncommitted code, for example by using git reset --soft HEAD~1?



Hypothetical answer (HyDE):
  To undo your last commit while keeping your changes, use the command `git reset --soft HEAD~1`. This moves the branch pointer back one commit while leaving the staged changes intact in the index, so your work remains ready to be recommitted. If you want the changes unstaged instead, use `git reset HEAD~1` (the default `--mixed` mode), which keeps your modifications in the working directory without



multi_query    gold rank: 4
   1. Git > Introduction > Removing files > Throwing away local, uncommitted changes
   2. Git > Introduction > Removing files > Reverting a commit
   3. Git > Introduction > Removing files
   4. ✓ Git > Introduction > Removing files > Resetting a commit
   5. Git > Introduction > Adding and committing files


hyde           gold rank: 6
   1. Git > Introduction > Removing files > Throwing away local, uncommitted changes
   2. Git > Introduction > Removing files
   3. Git > Introduction > Removing files > Reverting a commit
   4. Git > Rebasing > Principle
   5. Git > Introduction > Conclusion


## 4. Answering with citations

The top 5 sections (default method: `vector`, the best for answers in section 6) go to Claude (Haiku), numbered `[1]`–`[5]`, with rules: use only these sections,
cite every fact, and if the answer isn't there start with **"Not in docs:"**. The same rule as a
company-docs bot: cite or say you don't know.

In [13]:
from docsrag.answer import answer

def ask(question, method="vector"):
    r = answer(question, method)
    print(f"Q: {question}\n\n{r['answer']}\n\nSources:")
    for i, sid in enumerate(r["sources"], 1):
        s = ix.sections[sid]
        print(f"  {'*' if sid in r['cited'] else ' '}[{i}] {s['book_title']} > {s['page']} > {s['heading']}")
    print()

ask("How do I get the elements that are in either of two Python sets but not in both?")
ask("What is a good recipe for sourdough bread?")

Q: How do I get the elements that are in either of two Python sets but not in both?

Use the symmetric difference, either with the method `symmetric_difference()` or the `^` operator. It returns the elements that are in one set or the other, but not in both [5].

```python
s1 = set([4, 6, 9])
s2 = set([1, 6, 8])
s1.symmetric_difference(s2)   # set([8, 1, 4, 9])
s1 ^ s2                       # set([8, 1, 4, 9])
```

To update `s1` in place, use `s1.symmetric_difference_update(s2)` [5].

Sources:
   [1] Python Programming > Sets > Overview > Set Operations > Intersection
   [2] Python Programming > Sets > Overview > Set Operations > Set Difference
   [3] Python Programming > Lists > Difference
   [4] Python Programming > Sets > Overview > Multiple sets
  *[5] Python Programming > Sets > Overview > Set Operations > Symmetric Difference



Q: What is a good recipe for sourdough bread?

Not in docs: The provided sections cover SQL query ordering (ORDER BY), SQL NULL handling, relational database concepts, and Git-based Linux systems. None of them contain a recipe for sourdough bread, so I can't answer this question from the documentation.

Sources:
   [1] Structured Query Language > Data Query Language > SELECT query > ORDER BY clause
   [2] Structured Query Language > Data Query Language > SELECT query > First query
   [3] Structured Query Language > Relational Databases > Introduction
   [4] Structured Query Language > SELECT: IS NULL Predicate > Some Examples
   [5] Git > Systems > Git-based systems > Case Studies



## 5. The MCP server

`mcp_server.py` exposes the same pipeline to any MCP client (Claude Code, the Inspector…) and uses
each server feature from *AI Agents with MCP*:

| Feature | Here |
|---|---|
| Tools | `search_docs`, `get_section`, `refresh_book` (with **progress notifications**) |
| Resources | `doc://books`, and the template `doc://{book}/{page}` |
| Prompt | `answer_with_citations` |
| Completion | autocompletes book and page names for the resource template |
| Errors | `ToolError`, so the model reads what went wrong |

Below, a raw stdio session: the same JSON-RPC messages a client sends.

In [14]:
msgs = [
    {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18",
     "capabilities": {}, "clientInfo": {"name": "notebook", "version": "1"}}},
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 3, "method": "resources/templates/list"},
    {"jsonrpc": "2.0", "id": 4, "method": "prompts/list"},
    {"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "search_docs",
     "arguments": {"query": "insert or update a row in one statement", "k": 3}}},
    {"jsonrpc": "2.0", "id": 6, "method": "tools/call", "params": {"name": "get_section",
     "arguments": {"section_id": "sql/MERGE#does-not-exist"}}},
    {"jsonrpc": "2.0", "id": 7, "method": "completion/complete", "params": {
     "ref": {"type": "ref/resource", "uri": "doc://{book}/{page}"},
     "argument": {"name": "page", "value": "lis"}, "context": {"arguments": {"book": "python"}}}},
]
script = "import sys,time\n" + "".join(f"print({json.dumps(json.dumps(m))}, flush=True)\n" for m in msgs) + "time.sleep(25)\n"
feeder = subprocess.Popen([sys.executable, "-c", script], stdout=subprocess.PIPE)
server = subprocess.Popen([sys.executable, str(ROOT / "mcp_server.py")], stdin=feeder.stdout,
                          stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, cwd=ROOT)
replies = {}
while len(replies) < 7:
    line = server.stdout.readline()
    if not line:
        break
    d = json.loads(line)
    if "id" in d:
        replies[d["id"]] = d.get("result", d.get("error"))
server.kill(); feeder.kill()

print("tools:     ", [t["name"] for t in replies[2]["tools"]])
print("templates: ", [t["uriTemplate"] for t in replies[3]["resourceTemplates"]])
print("prompts:   ", [p["name"] for p in replies[4]["prompts"]])
print("\nsearch_docs ->\n" + replies[5]["content"][0]["text"])
print("\nget_section with a bad id -> isError =", replies[6].get("isError"))
print("  " + replies[6]["content"][0]["text"][:220])
print("\ncompletion for doc://python/lis... ->", replies[7]["completion"]["values"][:6])

tools:      ['search_docs', 'get_section', 'refresh_book']
templates:  ['doc://{book}/{page}']
prompts:    ['answer_with_citations']

search_docs ->
- sql/INSERT_1#introduction
  Structured Query Language > INSERT 1 > Introduction
  Hint: Be carefull and deactivate AUTOCOMMIT. The INSERT command stores one or more new rows into one table. The content of the new rows consists of either fixed values or results from a SELECT, which is evaluated at runt...
- sql/Data_Manipulation_Language#insert-statement
  Structured Query Language > Data Manipulation Language > INSERT statement
  The exhaustive syntax of the INSERT statement is as follows: The INSERT statement is used to add new records (rows) in a table. For instance, we want to add a new reunion: Its primary key is 7, Its name is "Job interview...
- sql/INSERT_2#evaluate-rows-at-runtime
  Structured Query Language > INSERT 2 > Evaluate Rows at Runtime
  Similar to the previous example of a single scalar value from a scalar value subque

## 6. Evaluation

### The test set

For a fixed random sample of sections, Claude (Sonnet) wrote the question a reader would ask
**in their own words**; that section is the gold label. Plus 6 hand-written **out-of-scope**
questions the bot must refuse.

In [15]:
questions = [json.loads(l) for l in open(ROOT / "eval/questions.jsonl")]
inscope = [x for x in questions if x["in_scope"]]
print(f"{len(questions)} questions: {len(inscope)} in scope "
      f"({sum(x['book']=='python' for x in inscope)} Python, {sum(x['book']=='sql' for x in inscope)} SQL, "
      f"{sum(x['book']=='git' for x in inscope)} Git) + {len(questions)-len(inscope)} out of scope\n")
table([[x["id"], x["question"][:95], x["gold_section"] or "—"] for x in questions[::7]],
      ["id", "Question", "Gold section"])

48 questions: 42 in scope (18 Python, 14 SQL, 10 Git) + 6 out of scope



| id | Question | Gold section |
|---|---|---|
| python-01 | How can I make a Python function give back more than one value at once, and how do I capture th | python/Functions#function-calls-defining-functions |
| python-08 | Which version control systems do most open source projects use, and why would I want to use one | python/Editing_and_Running_Python_Code#managing-code-version-control |
| python-15 | pip install fails for a package I need. How can I install it manually from a downloaded .whl fi | python/Package_management#introduction |
| sql-04 | What guarantees does a database give me when several users change data at the same time, and wh | sql/Database_Management_Systems_(DBMS)#historical-context-what-makes-up-a-database-management-system |
| sql-11 | What are the two kinds of INTERVAL in standard SQL, and which fields does each one hold? | sql/Data_Types#temporal |
| git-04 | What happens when I run git merge on another branch, and when does it make a merge commit versu | git/Branching_&_merging#merging |
| out-01 | How do I configure a VLAN on a Cisco switch? | — |

### Retrieval: which method finds the right section?

- **hit@1 / hit@5:** the gold section is ranked 1st / in the top 5
- **page@5:** a section of the gold *page* is in the top 5
- **MRR@10:** 1 / rank of the gold section (0 if not in the top 10), averaged

In [16]:
ret = json.load(open(ROOT / "eval/results/retrieval.json"))
S = ret["summary"]
table([[m, f"{s['hit@1']:.2f}", f"{s['hit@5']:.2f}", f"{s['page@5']:.2f}", f"{s['MRR@10']:.2f}",
        f"{s['avg_ms']} ms", s["llm_calls_per_q"]] for m, s in S.items()],
      ["Method", "hit@1", "hit@5", "page@5", "MRR@10", "Avg time", "Claude calls"])

| Method | hit@1 | hit@5 | page@5 | MRR@10 | Avg time | Claude calls |
|---|---|---|---|---|---|---|
| vector | 0.67 | 1.00 | 1.00 | 0.80 | 161 ms | 0.0 |
| bm25 | 0.69 | 0.88 | 1.00 | 0.77 | 2 ms | 0.0 |
| hybrid | 0.64 | 0.98 | 1.00 | 0.78 | 11 ms | 0.0 |
| hybrid_rerank | 0.71 | 0.93 | 1.00 | 0.82 | 475 ms | 0.0 |
| multi_query | 0.67 | 1.00 | 1.00 | 0.81 | 1663 ms | 1.0 |
| hyde | 0.67 | 1.00 | 1.00 | 0.81 | 2024 ms | 1.0 |

Where the methods **disagree** (gold rank per method; `—` = not in the top 10):

In [17]:
by = {}
for r in ret["rows"]:
    by.setdefault(r["id"], {})[r["method"]] = r["rank"]
rows = [[qid] + [d.get(m) or "—" for m in METHODS] for qid, d in by.items()
        if len({d.get(m) for m in METHODS}) > 1 and any(d.get(m) != 1 for m in METHODS)]
table(rows, ["id"] + METHODS)

| id | vector | bm25 | hybrid | hybrid_rerank | multi_query | hyde |
|---|---|---|---|---|---|---|
| python-01 | 1 | — | 3 | 2 | 2 | 2 |
| python-02 | 1 | 2 | 1 | 1 | 2 | 2 |
| python-03 | 2 | — | 6 | 8 | 2 | 5 |
| python-06 | 4 | 7 | 5 | 6 | 3 | 4 |
| python-07 | 5 | 1 | 2 | 1 | 1 | 1 |
| python-08 | 1 | 1 | 1 | 1 | 4 | 1 |
| python-11 | 2 | 1 | 2 | 1 | 2 | 2 |
| python-12 | 1 | 1 | 1 | 1 | 2 | 1 |
| python-13 | 2 | 1 | 1 | 2 | 3 | 2 |
| python-14 | 2 | 2 | 2 | 3 | 1 | 2 |
| sql-01 | 1 | 1 | 1 | 2 | 1 | 1 |
| sql-02 | 1 | — | 3 | 1 | 2 | 2 |
| sql-03 | 1 | — | 5 | 2 | 2 | 3 |
| sql-05 | 1 | 2 | 1 | 1 | 1 | 1 |
| sql-07 | 5 | 1 | 2 | 2 | 1 | 2 |
| sql-08 | 2 | 1 | 2 | 1 | 2 | 2 |
| sql-10 | 1 | 2 | 1 | 1 | 1 | 1 |
| sql-11 | 2 | 1 | 2 | 1 | 2 | 1 |
| sql-12 | 2 | 1 | 1 | 1 | 1 | 1 |
| sql-13 | 4 | 4 | 4 | 2 | 5 | 4 |
| git-02 | 1 | 3 | 1 | — | 1 | 1 |
| git-03 | 3 | 5 | 3 | 1 | 2 | 2 |
| git-07 | 3 | 1 | 2 | 4 | 1 | 1 |
| git-08 | 1 | 1 | 1 | 1 | 1 | 2 |
| git-09 | 2 | 3 | 2 | 2 | 1 | 1 |

### Answers: correct, grounded, cited, and refusing when it should

A judge model (Sonnet) compares each answer with the reference answer (**correct**) and checks every
claim against the retrieved sections (**grounded**).

In [18]:
ans = json.load(open(ROOT / "eval/results/answers.json"))
A = ans["summary"]
table([[m, f"{s['correct']:.2f}", f"{s['grounded']:.2f}", f"{s['cites_gold']:.2f}",
        s["refusals_out_of_scope"], s["false_refusals"]] for m, s in A.items()],
      ["Retrieval method", "Correct", "Grounded", "Cites gold", "Refused out-of-scope", "False refusals"])

| Retrieval method | Correct | Grounded | Cites gold | Refused out-of-scope | False refusals |
|---|---|---|---|---|---|
| vector | 0.95 | 0.81 | 1.00 | 6/6 | 0/42 |
| hybrid_rerank | 0.95 | 0.57 | 0.91 | 6/6 | 1/42 |

Every in-scope miss, for both methods: answered **"Not in docs"** (false refusal), judged incorrect, or not grounded:

In [19]:
qinfo = {x["id"]: x for x in questions}
for m in A:
    print(f"== {m}")
    for r in ans["rows"]:
        j = r.get("judge", {})
        if r["method"] == m and r["in_scope"] and (r["not_in_docs"] or not j.get("correct") or not j.get("grounded")):
            tags = [t for t, bad in [("NOT IN DOCS", r["not_in_docs"]), ("INCORRECT", not j.get("correct")),
                                     ("NOT GROUNDED", not j.get("grounded"))] if bad]
            found = "gold retrieved" if qinfo[r["id"]]["gold_section"] in r["sources"] else "gold NOT retrieved"
            print(f"  [{', '.join(tags)}] {r['id']} ({found}): {qinfo[r['id']]['question'][:80]}")
            print(f"      judge: {j.get('why', '')[:150]}")
    print()

== vector
  [NOT GROUNDED] python-11 (gold retrieved): Why does type(3) is object return False in Python even though isinstance(3, obje
      judge: Matches reference, but claims 'every class inherits from object', 'type(3) is int', and 'int is a subclass of object' are not stated in the retrieved 
  [INCORRECT] python-16 (gold retrieved): How can I use nested for-loops to print a 3x3 multiplication table, and how do I
      judge: Answer says the docs lack a solution (a 'Not in docs' reply) rather than conveying the reference info; its claims about the sections are accurate and 
  [NOT GROUNDED] python-18 (gold retrieved): How can I run a SQLite query with APSW in Python and loop over the rows one at a
      judge: Core answer matches the reference (cursor.execute iterator), but the code examples and the `?` placeholder syntax are not in the retrieved sections, s
  [NOT GROUNDED] sql-03 (gold retrieved): My query returns rows in a different order sometimes, and some rows have the sam


## 7. Findings

**1. Every method finds the right *page*; on these questions the methods are close.**
page@5 is 1.00 for every method. Re-ranking gives the best top result (hit@1 0.71, MRR 0.82); `vector`,
`multi_query` and `hyde` never miss the top 5. The two methods that call Claude (`multi_query`, `hyde`)
add ~1.7–2 s and a paid call per question without beating plain vector search here.

**2. Hybrid did not beat plain vector search, and re-ranking hurt the answers.**
BM25 misses questions phrased in everyday words, and fusion pulls those sections down (the
*"undo my last commit"* example in section 3). The cross-encoder raised hit@1 but dropped some correct
sections out of the top 5, and answers built on its results were **less grounded (0.57 vs 0.81)**, with
equal correctness (0.95). The re-ranker is trained on web search (MS MARCO), not manuals.
**So the default method is `vector`:** the best answers, no extra Claude calls.

**3. The main remaining weakness is grounding, not retrieval.** Answers are correct 95% of the time and
cite the exact gold section (vector: 100%), but about 1 in 5 adds "helpful" details that aren't in the
sources: extra example code, a table name, *"Python packs these into a tuple"*. The judge marks those
claims unsupported. **Next:** a stricter prompt (no code or facts beyond the sections), and a check that
every cited claim appears in its section.

**4. Refusing works.** All 6 off-topic questions were refused, and almost no real question was
(0/42 for vector, 1/42 for re-rank, where the right section wasn't retrieved).

**5. Evaluating the evaluator: two bugs found by looking at the outputs, not the averages.**
- **The judge** reasoned before its JSON and was cut off at 200 tokens; cut-off verdicts counted as
  failures (first scores: vector 0.48, re-rank 0.33 correct). Fixed with a JSON-first prompt and room to
  finish; saved answers were re-graded, not regenerated.
- **Claude's hidden "thinking" used up the token limits**, so HyDE passages, multi-query rephrasings and
  answers were often cut short or empty. That made HyDE look worst (MRR 0.73 → **0.81** once fixed) and
  multi-query look best (hit@1 0.76 → **0.67**), and made truncated answers look *more* grounded and
  produced false "Not in docs" replies. Fixed by turning thinking off for these short calls (models
  differ: `disabled` vs `between_tools`) and **counting every reply that hits the limit** (0 in the final run).

**Caveat:** the in-scope questions were written by Claude *from* each section, so they're specific and
favour vector search (it already reaches hit@5 = 1.00). Real users type shorter, keyword-heavy
questions (error messages, command names) where BM25 and hybrid usually help. **Next:** a
hand-written question set.

| | Best on this test set |
|---|---|
| Top-ranked section | `hybrid_rerank` (hit@1 0.71, MRR 0.82) |
| Never missing the top 5 | `vector`, `multi_query`, `hyde` (hit@5 1.00) |
| Answers (correct / grounded) | `vector` (0.95 / 0.81) |
| Speed | `bm25` (2 ms) |
| Refusing off-topic questions | all methods (6/6) |


In [20]:
print(f"Claude calls made by this notebook: {USAGE['calls']} "
      f"({USAGE['input_tokens']:,} input / {USAGE['output_tokens']:,} output tokens, {USAGE['truncated']} cut off)")

Claude calls made by this notebook: 6 (3,963 input / 786 output tokens, 0 cut off)
